# 08. CNN: U-Net 3D para el ventrículo lateral

Red neuronal convolucional 3D (MONAI) entrenada con HyKid, usando **la misma partición 70 / 10 / 20 por paciente** del notebook 06 para que la comparación sea justa.

- **Entrada**: `eT1W_FFE_SVR` (1 mm). **Salida**: máscara binaria del ventrículo lateral (`gt == 4`).
- **Preprocesamiento**: orientación RAS, recorte al cerebro, normalización z-score dentro del cerebro.
- **Aumento de datos**: parches 96³ con muestreo positivo/negativo, flips, rotaciones y escalas pequeñas, cambios de intensidad.
- **Modelo**: U-Net residual (16-32-64-128-256), pérdida Dice + entropía cruzada, AdamW.
- **Inferencia**: ventana deslizante + *test-time augmentation* (flips) que además da un **mapa de incertidumbre**.
- **Evaluación**: Dice, error de volumen y HD95 en test, comparado con SynthSeg, clásico e híbrido.

**Se recomienda Google Colab con GPU** (Entorno de ejecución > Cambiar tipo > GPU T4). En CPU es inviable.

Las predicciones se guardan en `predicciones/UNet3D/` en el espacio original, así el notebook 04 las incluye solo.

## 0. Entorno (Colab)
Sube la carpeta del proyecto a tu Google Drive (con `data/raw` y `outputs/figuras_06_final/particion.csv`) y ajusta `PROYECTO`.

In [ ]:
import sys, subprocess
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "monai[nibabel]", "scikit-image"], check=True)

In [ ]:
from pathlib import Path

PROYECTO = Path("/content/drive/MyDrive/hykid") if EN_COLAB else \
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "paths.py").exists())
SUBJECTS = PROYECTO / "data" / "raw" / "data"
OUTPUTS = PROYECTO / "outputs"
PARTICION = OUTPUTS / "figuras_06_final" / "particion.csv"
BASE_METRICAS = OUTPUTS / "figuras_06_final" / "base_metricas.csv"
FIG_DIR = OUTPUTS / "figuras_08_unet"
PRED_DIR = OUTPUTS / "predicciones" / "UNet3D"
CKPT = OUTPUTS / "modelos" / "unet3d_ventriculo.pt"
ESTADO = OUTPUTS / "modelos" / "unet3d_estado.pt"
for d in [FIG_DIR, PRED_DIR, CKPT.parent]:
    d.mkdir(parents=True, exist_ok=True)
print(PROYECTO, SUBJECTS.exists(), PARTICION.exists())

## 1. Configuración

In [ ]:
import time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib
import torch
from scipy import ndimage as ndi

from monai.data import Dataset, CacheDataset, DataLoader, decollate_batch
from monai.inferers import sliding_window_inference
from monai.losses import DiceCELoss
from monai.networks.nets import UNet
from monai.utils import set_determinism
from monai.transforms import (Compose, LoadImaged, EnsureChannelFirstd, Orientationd, CropForegroundd,
                              NormalizeIntensityd, RandCropByPosNegLabeld, RandFlipd, RandAffined,
                              RandScaleIntensityd, RandShiftIntensityd, Lambdad, EnsureTyped, SpatialPadd)

SEED = 42
ROI = (96, 96, 96)
BATCH = 2
MUESTRAS_POR_VOLUMEN = 4
MAX_EPOCHS = 500
GUARDAR_CADA = 1
CONTINUAR_DESDE_PREVIO = True
VAL_CADA = 10
LR = 2e-4
ENTRENAR = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_determinism(SEED)
print("Dispositivo:", device)

## 2. Datos y partición

In [ ]:
part = pd.read_csv(PARTICION)

def item(r):
    s = SUBJECTS / r.subject / r.scan
    return {"image": str(s / "eT1W_FFE_SVR.nii.gz"), "label": str(s / "segmentation" / "gt.nii.gz"),
            "subject": r.subject, "scan": r.scan}

datos = {k: [item(r) for r in part[part.split == k].itertuples()
             if (SUBJECTS / r.subject / r.scan / "eT1W_FFE_SVR.nii.gz").exists()]
         for k in ["train", "val", "test"]}
print({k: len(v) for k, v in datos.items()})

In [ ]:
solo_ventriculo = Lambdad(keys="label", func=lambda x: (x == 4).astype(np.float32))
base_tf = [
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    solo_ventriculo,
    CropForegroundd(keys=["image", "label"], source_key="image", margin=4),
    NormalizeIntensityd(keys="image", nonzero=True),
    SpatialPadd(keys=["image", "label"], spatial_size=ROI),
]
train_tf = Compose(base_tf + [
    RandCropByPosNegLabeld(keys=["image", "label"], label_key="label", spatial_size=ROI,
                           pos=2, neg=1, num_samples=MUESTRAS_POR_VOLUMEN, image_key="image", image_threshold=0),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
    RandAffined(keys=["image", "label"], prob=0.3, rotate_range=(0.15,) * 3, scale_range=(0.1,) * 3,
                mode=("bilinear", "nearest"), padding_mode="zeros"),
    RandScaleIntensityd(keys="image", factors=0.1, prob=0.5),
    RandShiftIntensityd(keys="image", offsets=0.1, prob=0.5),
    EnsureTyped(keys=["image", "label"]),
])
eval_tf = Compose(base_tf + [EnsureTyped(keys=["image", "label"])])

NW = 0 if sys.platform == "win32" else 2
train_ds = CacheDataset(datos["train"], train_tf, cache_rate=1.0, num_workers=NW)
val_ds = CacheDataset(datos["val"], eval_tf, cache_rate=1.0, num_workers=NW)
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=NW)
val_dl = DataLoader(val_ds, batch_size=1, num_workers=NW)

## 3. Modelo y entrenamiento

In [ ]:
model = UNet(spatial_dims=3, in_channels=1, out_channels=2, channels=(16, 32, 64, 128, 256),
             strides=(2, 2, 2, 2), num_res_units=2, dropout=0.1).to(device)
loss_fn = DiceCELoss(to_onehot_y=True, softmax=True)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCHS)
scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
print(sum(p.numel() for p in model.parameters()) / 1e6, "M parámetros")


def dice_np(a, b):
    d = a.sum() + b.sum()
    return 2 * (a & b).sum() / d if d else np.nan


def predecir(img_t, tta=False):
    model.eval()
    with torch.no_grad(), torch.autocast(device.type, enabled=device.type == "cuda"):
        f = lambda x: torch.softmax(sliding_window_inference(x, ROI, 4, model, overlap=0.5), 1)
        p = f(img_t)
        if tta:
            ps = [p]
            for ax in [2, 3, 4]:
                ps.append(torch.flip(f(torch.flip(img_t, [ax])), [ax]))
            p = torch.stack(ps).mean(0)
    return p

### Entrenamiento con autoguardado

Cada época se guarda `unet3d_estado.pt` (modelo, optimizador, scheduler, época, historial, mejor Dice). Si se interrumpe (Colab se desconecta, cierras el notebook), al volver a correr esta celda **continúa desde la última época guardada**.

- Si no hay estado pero existe el modelo previo `unet3d_ventriculo.pt` y `CONTINUAR_DESDE_PREVIO = True`, arranca con esos pesos (no desde cero).
- Para empezar de cero: borrar `unet3d_estado.pt` y poner `CONTINUAR_DESDE_PREVIO = False`.

In [ ]:
def guardar_estado(ep, hist, mejor):
    tmp = ESTADO.with_suffix(".tmp")
    torch.save({"epoch": ep, "model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                "scaler": scaler.state_dict(), "hist": hist, "mejor": mejor, "max_epochs": MAX_EPOCHS}, tmp)
    tmp.replace(ESTADO)


inicio, mejor = 1, -1.0
hist = {"epoch": [], "loss": [], "val_dice": []}
if ESTADO.exists():
    st = torch.load(ESTADO, map_location=device, weights_only=False)
    model.load_state_dict(st["model"]); opt.load_state_dict(st["opt"])
    sched.load_state_dict(st["sched"]); scaler.load_state_dict(st["scaler"])
    inicio, hist, mejor = st["epoch"] + 1, st["hist"], st["mejor"]
    print(f"Reanudando desde la época {inicio} (mejor Dice val {mejor:.3f})")
elif CKPT.exists() and CONTINUAR_DESDE_PREVIO:
    model.load_state_dict(torch.load(CKPT, map_location=device))
    print("Sin estado guardado: se inicia con los pesos previos de", CKPT.name)
else:
    print("Entrenamiento desde cero")

if ENTRENAR and inicio <= MAX_EPOCHS:
    t0 = time.time()
    for ep in range(inicio, MAX_EPOCHS + 1):
        model.train()
        perdidas = []
        for b in train_dl:
            x, y = b["image"].to(device), b["label"].to(device)
            opt.zero_grad()
            with torch.autocast(device.type, enabled=device.type == "cuda"):
                loss = loss_fn(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            perdidas.append(loss.item())
        sched.step()
        vd = np.nan
        if ep % VAL_CADA == 0 or ep == MAX_EPOCHS:
            ds = []
            for b in val_dl:
                p = predecir(b["image"].to(device)).argmax(1).cpu().numpy()[0].astype(bool)
                ds.append(dice_np(p, b["label"].numpy()[0, 0].astype(bool)))
            vd = float(np.nanmean(ds))
            if vd > mejor:
                mejor = vd
                torch.save(model.state_dict(), CKPT)
            print(f"época {ep:3d} | pérdida {np.mean(perdidas):.4f} | Dice val {vd:.3f} | mejor {mejor:.3f} | {(time.time() - t0) / 60:.1f} min")
        hist["epoch"].append(ep); hist["loss"].append(float(np.mean(perdidas))); hist["val_dice"].append(vd)
        if ep % GUARDAR_CADA == 0 or ep == MAX_EPOCHS:
            guardar_estado(ep, hist, mejor)
    pd.DataFrame(hist).to_csv(FIG_DIR / "historial_entrenamiento.csv", index=False)
elif inicio > MAX_EPOCHS:
    print("Entrenamiento ya completo")

model.load_state_dict(torch.load(CKPT, map_location=device))
print("Modelo cargado:", CKPT)

In [ ]:
if hist["epoch"]:
    h = pd.DataFrame(hist)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(h.epoch, h.loss); ax[0].set_xlabel("época"); ax[0].set_ylabel("pérdida"); ax[0].set_title("Entrenamiento")
    v = h.dropna()
    ax[1].plot(v.epoch, v.val_dice, "o-"); ax[1].set_xlabel("época"); ax[1].set_ylabel("Dice"); ax[1].set_title("Validación")
    plt.tight_layout(); plt.savefig(FIG_DIR / "curvas_entrenamiento.png", dpi=110, bbox_inches="tight"); plt.show()

## 4. Inferencia en test (espacio original) + incertidumbre por TTA
La incertidumbre es la entropía media de la probabilidad dentro de la región predicha: alta = el modelo duda.

In [ ]:
def hd95(a, b, zooms):
    if a.sum() == 0 or b.sum() == 0:
        return np.nan
    ba, bb = a & ~ndi.binary_erosion(a), b & ~ndi.binary_erosion(b)
    da = ndi.distance_transform_edt(~bb, sampling=zooms)[ba]
    db = ndi.distance_transform_edt(~ba, sampling=zooms)[bb]
    return np.percentile(np.concatenate([da, db]), 95)


def dos_mayores(m):
    lab, n = ndi.label(m)
    if n <= 2:
        return m
    sz = ndi.sum(m, lab, range(1, n + 1))
    return np.isin(lab, np.argsort(sz)[::-1][:2] + 1)


def inferir(it):
    t0 = time.time()
    ref = nib.as_closest_canonical(nib.load(it["image"]))
    zooms = ref.header.get_zooms()[:3]
    g = np.asarray(nib.as_closest_canonical(nib.load(it["label"])).dataobj).astype(int) == 4

    img = ref.get_fdata().astype(np.float32)
    fg = np.where(img > 0)
    box = tuple(slice(max(i.min() - 4, 0), i.max() + 5) for i in fg)
    crop = img[box]
    m_ = crop > 0
    crop = np.where(m_, (crop - crop[m_].mean()) / (crop[m_].std() + 1e-8), 0).astype(np.float32)
    x = torch.from_numpy(crop)[None, None].to(device)
    pad = [max(r - s, 0) for r, s in zip(ROI, crop.shape)]
    x = torch.nn.functional.pad(x, (0, pad[2], 0, pad[1], 0, pad[0]))

    prob = predecir(x, tta=True)[0, 1].float().cpu().numpy()[:crop.shape[0], :crop.shape[1], :crop.shape[2]]
    pred = np.zeros(g.shape, bool)
    pred[box] = dos_mayores(prob > 0.5)
    pfull = np.zeros(g.shape, np.float32)
    pfull[box] = prob
    ent = -(pfull * np.log(pfull + 1e-8) + (1 - pfull) * np.log(1 - pfull + 1e-8))
    region = ndi.binary_dilation(pred, iterations=2) if pred.any() else pfull > 0.1

    nib.save(nib.Nifti1Image(pred.astype(np.uint8), ref.affine), PRED_DIR / f"{it['subject']}_{it['scan']}.nii.gz")
    ml = float(np.prod(zooms)) / 1000
    fila = {"subject": it["subject"], "scan": it["scan"], "vol_gt": g.sum() * ml,
            "dice_UNet3D": dice_np(pred, g), "vol_UNet3D": pred.sum() * ml, "hd95_UNet3D": hd95(pred, g, zooms),
            "incertidumbre": float(ent[region].mean()) if region.any() else np.nan}
    print(f"{it['subject']}/{it['scan']} ({time.time() - t0:.0f}s): Dice {fila['dice_UNet3D']:.3f} | "
          f"{fila['vol_UNet3D']:.0f} vs {fila['vol_gt']:.0f} mL | incertidumbre {fila['incertidumbre']:.3f}")
    return fila, (img, g, pred, ent)


ejemplos, filas_val, filas = {}, [], []
print("Validación:")
for it in datos["val"]:
    f, _ = inferir(it)
    filas_val.append(f)
print("Test:")
for it in datos["test"]:
    f, ej = inferir(it)
    filas.append(f)
    ejemplos[(it["subject"], it["scan"])] = ej
unet_val = pd.DataFrame(filas_val)
unet = pd.DataFrame(filas)

## 5. Comparación en test con SynthSeg, clásico e híbrido

In [ ]:
from scipy.stats import wilcoxon, spearmanr

comp = unet.copy()
if BASE_METRICAS.exists():
    b = pd.read_csv(BASE_METRICAS)
    comp = comp.merge(b[["subject", "scan", "dice_SynthSeg", "vol_SynthSeg", "hd95_SynthSeg",
                         "dice_Clasico", "vol_Clasico", "hd95_Clasico"]], on=["subject", "scan"], how="left")
    usa_cl = comp.vol_Clasico > 200
    for met in ["dice", "vol", "hd95"]:
        comp[f"{met}_Hibrido"] = np.where(usa_cl, comp[f"{met}_Clasico"], comp[f"{met}_SynthSeg"])
METODOS = [m for m in ["SynthSeg", "Clasico", "Hibrido", "UNet3D"] if f"dice_{m}" in comp]
comp["rango"] = pd.cut(comp.vol_gt, [0, 30, 150, np.inf], labels=["pequeño", "mediano", "grande"])
comp.to_csv(FIG_DIR / "test_unet_comparacion.csv", index=False)

rows = []
for grupo, sub in [("total", comp)] + list(comp.groupby("rango", observed=True)):
    for m in METODOS:
        d = sub[f"vol_{m}"] - sub.vol_gt
        rows.append({"rango": grupo, "metodo": m, "n": len(sub), "dice_media": sub[f"dice_{m}"].mean(),
                     "dice_mediana": sub[f"dice_{m}"].median(), "dice_min": sub[f"dice_{m}"].min(),
                     "err_vol_abs_mediana_%": (d / sub.vol_gt * 100).abs().median(),
                     "hd95_mediana_mm": sub[f"hd95_{m}"].median(), "sesgo_ml": d.mean(), "LoA_ml": 1.96 * d.std()})
res = pd.DataFrame(rows)
res.to_csv(FIG_DIR / "resultados_test_unet.csv", index=False)
display(res.round(3))

for m in METODOS:
    if m == "UNet3D":
        continue
    try:
        p = wilcoxon(comp.dice_UNet3D, comp[f"dice_{m}"]).pvalue
    except ValueError:
        p = np.nan
    print(f"U-Net vs {m}: diferencia media {np.mean(comp.dice_UNet3D - comp[f'dice_{m}']):+.3f}, p = {p:.3f}")
rho, p = spearmanr(comp.incertidumbre, comp.dice_UNet3D)
print(f"Incertidumbre vs Dice: Spearman rho = {rho:.2f}, p = {p:.3f} (negativo = la incertidumbre anticipa errores)")

In [ ]:
col = {"SynthSeg": "tab:blue", "Clasico": "tab:orange", "Hibrido": "tab:red", "UNet3D": "tab:green"}
fig, ax = plt.subplots(1, 3, figsize=(19, 5))
bp = ax[0].boxplot([comp[f"dice_{m}"] for m in METODOS], patch_artist=True)
for bx, m in zip(bp["boxes"], METODOS):
    bx.set_facecolor(col[m])
ax[0].set_xticks(range(1, len(METODOS) + 1)); ax[0].set_xticklabels(METODOS)
ax[0].set_ylim(0, 1); ax[0].set_ylabel("Dice"); ax[0].set_title(f"Test (n={len(comp)})")
for m in METODOS:
    ax[1].scatter(comp.vol_gt, comp[f"dice_{m}"], s=28, color=col[m], label=m, alpha=0.85)
ax[1].set_xscale("log"); ax[1].set_ylim(0, 1); ax[1].legend()
ax[1].set_xlabel("volumen experto (mL)"); ax[1].set_ylabel("Dice"); ax[1].set_title("Dice según severidad")
ax[2].scatter(comp.incertidumbre, comp.dice_UNet3D, color="tab:green")
for _, r in comp.iterrows():
    ax[2].annotate(r.subject, (r.incertidumbre, r.dice_UNet3D), fontsize=7)
ax[2].set_xlabel("incertidumbre media (entropía)"); ax[2].set_ylabel("Dice U-Net")
ax[2].set_title("¿La incertidumbre anticipa el error?")
plt.tight_layout(); plt.savefig(FIG_DIR / "test_unet_resumen.png", dpi=110, bbox_inches="tight"); plt.show()

## 6. Casos de test: predicción e incertidumbre

In [ ]:
orden = comp.sort_values("vol_gt")
fig, axes = plt.subplots(len(orden), 3, figsize=(11, 3.5 * len(orden)), squeeze=False)
for row, (_, r) in zip(axes, orden.iterrows()):
    img, g, pred, ent = ejemplos[(r.subject, r.scan)]
    k = int(np.argmax(g.sum(axis=(0, 1)))) if g.any() else img.shape[2] // 2
    sl = lambda a: np.rot90(a[:, :, k])
    row[0].imshow(sl(img), cmap="gray"); row[0].contourf(sl(g), levels=[0.5, 1.5], colors=["purple"], alpha=0.45)
    row[0].set_title(f"{r.subject} experto {r.vol_gt:.0f} mL", fontsize=9)
    row[1].imshow(sl(img), cmap="gray"); row[1].contourf(sl(pred), levels=[0.5, 1.5], colors=["tab:green"], alpha=0.45)
    row[1].contour(sl(g), levels=[0.5], colors="white", linewidths=0.6)
    row[1].set_title(f"U-Net Dice {r.dice_UNet3D:.2f}", fontsize=9)
    row[2].imshow(sl(img), cmap="gray"); row[2].imshow(sl(ent), cmap="hot", alpha=0.6, vmin=0, vmax=0.7)
    row[2].set_title(f"incertidumbre {r.incertidumbre:.3f}", fontsize=9)
    for a in row:
        a.axis("off")
plt.tight_layout(); plt.savefig(FIG_DIR / "test_unet_casos.png", dpi=110, bbox_inches="tight"); plt.show()
print("Archivos en", FIG_DIR)
for f in sorted(FIG_DIR.iterdir()):
    print(" ", f.name)

## 7. Híbrido guiado por incertidumbre (U-Net + SynthSeg)

Regla: usar la U-Net; si su incertidumbre supera `U_MAX`, usar SynthSeg. `U_MAX` se **calibra en validación** (no en test). El barrido en test es solo análisis de sensibilidad.

In [ ]:
if BASE_METRICAS.exists():
    b = pd.read_csv(BASE_METRICAS)[["subject", "scan", "dice_SynthSeg", "vol_SynthSeg", "hd95_SynthSeg"]]
    v = unet_val.merge(b, on=["subject", "scan"], how="left")
    candidatos = np.round(np.arange(0.15, 0.71, 0.01), 2)

    def aplicar_u(df, u):
        usa_ss = df.incertidumbre > u
        out = {}
        for met in ["dice", "vol", "hd95"]:
            out[met] = np.where(usa_ss, df[f"{met}_SynthSeg"], df[f"{met}_UNet3D"])
        return out, usa_ss

    dv = [aplicar_u(v, u)[0]["dice"].mean() for u in candidatos]
    U_MAX = float(candidatos[int(np.argmax(dv))])
    print(f"U_MAX calibrado en validación (n={len(v)}): {U_MAX:.2f} | Dice val {max(dv):.3f}")

    out, usa_ss = aplicar_u(comp, U_MAX)
    comp["dice_UNet+SS"], comp["vol_UNet+SS"], comp["hd95_UNet+SS"] = out["dice"], out["vol"], out["hd95"]
    comp["usa_SynthSeg"] = usa_ss
    comp.to_csv(FIG_DIR / "test_unet_comparacion.csv", index=False)

    M2 = METODOS + ["UNet+SS"]
    filas_h = []
    for grupo, sub in [("total", comp)] + list(comp.groupby("rango", observed=True)):
        for m in M2:
            d = sub[f"vol_{m}"] - sub.vol_gt
            filas_h.append({"rango": grupo, "metodo": m, "n": len(sub), "dice_media": sub[f"dice_{m}"].mean(),
                            "dice_min": sub[f"dice_{m}"].min(),
                            "err_vol_abs_mediana_%": (d / sub.vol_gt * 100).abs().median(),
                            "sesgo_ml": d.mean(), "LoA_ml": 1.96 * d.std()})
    res_h = pd.DataFrame(filas_h)
    res_h.to_csv(FIG_DIR / "resultados_hibrido_incertidumbre.csv", index=False)
    display(res_h.round(3))
    display(comp[["subject", "vol_gt", "incertidumbre", "usa_SynthSeg", "dice_UNet3D", "dice_SynthSeg", "dice_UNet+SS"]].round(3))

    sens = pd.DataFrame({"U_MAX": candidatos,
                         "dice_val": dv,
                         "dice_test": [aplicar_u(comp, u)[0]["dice"].mean() for u in candidatos]})
    sens.to_csv(FIG_DIR / "sensibilidad_U_MAX.csv", index=False)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(sens.U_MAX, sens.dice_val, "o-", label="validación (calibra)")
    ax.plot(sens.U_MAX, sens.dice_test, "s--", label="test (sensibilidad)")
    ax.axvline(U_MAX, color="k", ls=":")
    ax.axhline(comp.dice_SynthSeg.mean(), color="tab:blue", ls=":", label="SynthSeg test")
    ax.axhline(comp.dice_UNet3D.mean(), color="tab:green", ls=":", label="U-Net test")
    ax.set_xlabel("U_MAX"); ax.set_ylabel("Dice medio"); ax.legend(fontsize=8)
    ax.set_title("Híbrido U-Net + SynthSeg según umbral de incertidumbre")
    plt.tight_layout(); plt.savefig(FIG_DIR / "hibrido_incertidumbre.png", dpi=110, bbox_inches="tight"); plt.show()

## Notas

- Con ~33 estudios de entrenamiento la red puede sobreajustar; el aumento de datos y el parcheo ayudan. Reportar el número de casos.
- El test tiene solo ~9 estudios (ver notebook 06): los resultados son orientativos. Para una evaluación más robusta se puede repetir el entrenamiento en 5 folds (5 veces más tiempo de GPU).
- `ENTRENAR = False` carga el modelo guardado y solo hace la inferencia.
- Con solo 4 casos de validación, `U_MAX` es una calibración débil: reportarlo como exploratorio.